In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install transformers datasets peft trl bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━ 204.8/528.8 kB 6.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/60.7 MB ? eta -:--:--

   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/60.7 MB 107.8 MB/s eta 0:00:01

   ━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/60.7 MB 158.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/60.7 MB 156.1 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 156.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 156.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 156.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 156.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.9 MB/s eta 0:00:00


# Step 1 - Imports

In [3]:
import torch
from datasets import load_dataset
from transformers import(
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import(
    LoraConfig,
    get_peft_model,
    TaskType,
)
from trl import SFTTrainer


# Step 2— Config

In [4]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME = "Kamran-56/prompt-refinement-dataset"
NEW_MODEL    = "Kamran-56/Qwen2.5-3B-PromptRefiner"

# Step 3 — Login to HuggingFace

In [5]:
from huggingface_hub import login

login(token = HF_TOKEN)

# Step 4 — Load Dataset

In [6]:
dataset = load_dataset(DATASET_NAME)
print(dataset['train'][0])

README.md: 0.00B [00:00, ?B/s]

prompt-refinement-dataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1561 [00:00<?, ? examples/s]

{'input': 'Write a personal essay of at least 1000 words discussing how embracing vulnerability and authenticity has affected your life. Use specific examples from your own experiences to support your arguments and make sure to address the following questions:', 'output': 'Assume the role of a reflective individual who has undergone significant personal growth, and write a heartfelt, 1000-word personal essay in a narrative, first-person format, discussing the transformative impact of embracing vulnerability and authenticity on your life. Adopt a sincere and introspective tone, using vivid storytelling and specific, detailed examples from your own experiences to illustrate the ways in which vulnerability and authenticity have shaped your relationships, decision-making, and overall well-being. \n\nIn your essay, address the following questions: How did you initially resist or struggle with being vulnerable, and what catalysts or events prompted you to adopt a more authentic approach to l

# Step 5 — Format Dataset into Chat Template


In [7]:
def format_row(row):
  return{
        "text": f"""<|im_start|>system
You are an expert prompt engineer. Transform the given basic prompt into a high quality, detailed and effective prompt.<|im_end|>
<|im_start|>user
{row["input"]}<|im_end|>
<|im_start|>assistant
{row["output"]}<|im_end|>"""
    }

dataset = dataset.map(format_row)
print("Formatted data: \n")
print(dataset["train"][0]["text"])

Map:   0%|          | 0/1561 [00:00<?, ? examples/s]

Formatted data: 

<|im_start|>system
You are an expert prompt engineer. Transform the given basic prompt into a high quality, detailed and effective prompt.<|im_end|>
<|im_start|>user
Write a personal essay of at least 1000 words discussing how embracing vulnerability and authenticity has affected your life. Use specific examples from your own experiences to support your arguments and make sure to address the following questions:<|im_end|>
<|im_start|>assistant
Assume the role of a reflective individual who has undergone significant personal growth, and write a heartfelt, 1000-word personal essay in a narrative, first-person format, discussing the transformative impact of embracing vulnerability and authenticity on your life. Adopt a sincere and introspective tone, using vivid storytelling and specific, detailed examples from your own experiences to illustrate the ways in which vulnerability and authenticity have shaped your relationships, decision-making, and overall well-being. 

In 

# Step 6 — Load Tokenizer


In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code = True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Step 7 — 4-bit Quantization Config

In [9]:
# Without quantization, Qwen 3B needs ~12GB VRAM
# With 4-bit quantization, it only needs ~4GB VRAM
# This is what makes it possible to run on free Colab T4

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # load model in 4-bit precision
    bnb_4bit_quant_type="nf4",       # nf4 is the best 4-bit format
    bnb_4bit_compute_dtype=torch.float16,  # use float16 for calculations
    bnb_4bit_use_double_quant=True,  # quantize the quantization itself (saves more memory)
)

# Step 8 — Load Model

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map = "auto",
    trust_remote_code = True,
)

# This line prepares the model for training with quantization
# Without this, gradients won't flow properly during training
model.config.use_cache = False               # disable cache during training
model.config.pretraining_tp = 1             # tensor parallelism = 1 (single GPU)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Step 9 — LoRA Config

In [11]:
# LoRA adds small trainable matrices to specific layers
# Instead of training 3 billion parameters
# We only train ~2-3 million LoRA parameters
# Much faster, much less memory, same quality

lora_config = LoraConfig(
    r=8,                     # rank — size of LoRA matrices
                              # higher = more capacity but more memory
                              # 16 is sweet spot for your task

    lora_alpha=16,            # scaling factor = r*2 is standard practice
                              # controls how much LoRA affects the model

    target_modules=[          # which layers to apply LoRA to
        "q_proj",             # query projection
        "k_proj",             # key projection
        "v_proj",             # value projection
        "o_proj",             # output projection
    ],                        # these are the attention layers in Qwen

    lora_dropout=0.05,        # randomly disable 5% of LoRA weights during training
                              # prevents overfitting on your small dataset

    bias="none",              # don't train bias parameters
    task_type=TaskType.CAUSAL_LM  # we are doing causal language modeling
)

# Attach LoRA adapter to the model
model = get_peft_model(model, lora_config)

# Show how many parameters we're actually training
model.print_trainable_parameters()
# Output will be something like:
# trainable params: 2,359,296 || all params: 3,102,654,464 || trainable%: 0.076
# Only 0.076% of parameters are being trained!

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


# Step 10 — Training Arguments

In [12]:
# All settings that control how training runs

training_args = TrainingArguments(
    output_dir="./results",          # where to save checkpoints locally

    num_train_epochs=3,              # go through entire dataset 3 times
                                     # 3 is good for 1561 rows, more might overfit

    per_device_train_batch_size=1,   # process 4 rows at a time
                                     # higher = faster but more VRAM
                                     # 4 is safe for Colab T4

    gradient_accumulation_steps=16,  # accumulate gradients over 4 steps
                                     # effective batch size = 4*4 = 16
                                     # simulates larger batch without more VRAM

    learning_rate=5e-5,              # how fast the model learns
                                     # 2e-4 is standard for LoRA fine-tuning

    fp16=False,       # ← turn this OFF
    bf16=True,        # ← turn this ON (Qwen uses BFloat16 natively)

    logging_steps=10,                # print training loss every 10 steps
                                     # so you can see progress

    save_steps=100,                  # save checkpoint every 100 steps
                                     # so you don't lose progress if crash

    warmup_ratio=0.03,               # slowly increase learning rate at start
                                     # for first 3% of training steps
                                     # prevents unstable training at beginning

    lr_scheduler_type="cosine",      # gradually reduce learning rate over time
                                     # cosine decay is smooth and works well

    report_to="none",                # don't report to wandb or tensorboard
                                     # keeps things simple
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# Step 11 — Create Trainer and Train

In [13]:
# SFTTrainer (Supervised Fine-Tuning Trainer)
# handles the entire training loop for us
# we just configure it and call .train()
def formatting_prompts_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,                     # our Qwen model with LoRA
    train_dataset=dataset["train"],  # your 1561 formatted rows
    processing_class=tokenizer,             # to convert text to numbers
    args=training_args,              # all our training settings
    formatting_func=formatting_prompts_func,     # the column that has formatted text
)

# This one line runs the entire training
# You will see loss printed every 10 steps
# Loss should go DOWN over time — that means model is learning
# Good final loss for your task: below 1.0

print("Starting training...")
trainer.train()
print("Training complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/1561 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1561 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1561 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1561 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


Step,Training Loss
10,2.437882
20,2.059523
30,1.870495
40,1.760476
50,1.667645
60,1.559983
70,1.527171
80,1.380306
90,1.396963
100,1.422043


In [ ]:
# Save the LoRA adapter weights locally first
trainer.model.save_pretrained("prompt-booster-3b")
tokenizer.save_pretrained("prompt-booster-3b")

# Push fine-tuned model to your HuggingFace Hub
# This will be at: huggingface.co/your-username/prompt-booster-3b
trainer.model.push_to_hub(NEW_MODEL, token=HF_TOKEN)
tokenizer.push_to_hub(NEW_MODEL, token=HF_TOKEN)

print(f"✅ Model pushed to huggingface.co/{NEW_MODEL}")